# Кейс №22. Пики и контр-пики (ML) — предобработка данных

**Заказчик:** аналитики и игроки, принимающие решения в драфте Dota 2.  
**Бизнес-вопрос:** влияют ли сочетания героев (пики и контр-пики) на вероятность победы команды и насколько сильно?

**Единица анализа:** матч (`match_id`) и пары героев Radiant × Dire.  
**Целевая переменная:** `radiant_win` — победа Radiant.  
**Важно:** этот ноутбук подготавливает признаки и обучающую выборку. Исторические win rate героев/пар нельзя рассчитывать на всей выборке до разбиения train/test — это приведёт к утечке целевой переменной.

Все таблицы кейса используют префикс `practice_case_22_`.

## 1. Методика и архитектура

- **RAW:** копия нужных полей из накопленной истории Lakehouse. Сырые строки сохраняются отдельно от очищенных.
- **STAGE:** типизация, дедупликация, проверка 10 игроков, валидности `hero_id`, `player_slot` и ссылочной целостности. Здесь нет ML-признаков и агрегатов win rate.
- **MART:** одна строка на матч с составами Radiant/Dire, целевой переменной и контекстом матча; отдельная таблица 5×5 пар героев для анализа контр-пиков.

### Гипотезы кейса
- **H1.** Состав героев содержит статистически значимый сигнал о победителе матча.
- **H2.** Некоторые пары героев имеют устойчивое преимущество/невыгодный матчап.
- **H3.** Добавление контекстных признаков (режим, длительность, ранг/уровень игроков — если доступны) улучшает качество модели относительно модели только по пикам.

### Ограничения
1. `start_time`, `duration`, `lobby_type`, `game_mode`, `cluster` используются только если присутствуют в источнике.
2. `is_radiant` берётся из источника; если отсутствует, сторона определяется по `player_slot < 128`.
3. Порядок пиков/банов не восстанавливается, если в источнике нет pick order / draft events.
4. Неизвестные герои и неполные драфты не должны молча превращаться в валидные наблюдения.

## 2. Источники и проверка схем

Ожидаемые источники накопленной истории:
- `lakehouse.raw.pg__matches`
- `lakehouse.raw.pg__match_players`
- справочник героев (имя, атрибут, тип атаки, роли) — имя таблицы уточняется по фактической схеме Lakehouse.

Перед загрузкой выводим наличие таблиц и их поля. Если фактическое имя справочника отличается, укажи его в `HEROES_SOURCE`.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from lakehouse_spark import SparkSessionContext
import pandas as pd
import time

CASE = "practice_case_22"
MATCHES_SOURCE = "lakehouse.raw.pg__matches"
PLAYERS_SOURCE = "lakehouse.raw.pg__match_players"
# Уточнить по каталогу Lakehouse. Можно оставить None, если справочник пока недоступен.
HEROES_SOURCE = "lakehouse.raw.pg__heroes"

RAW_MATCHES = f"lakehouse.raw.{CASE}_matches"
RAW_PLAYERS = f"lakehouse.raw.{CASE}_match_players"
RAW_HEROES = f"lakehouse.raw.{CASE}_heroes"

STAGE_MATCHES = f"lakehouse.stage.{CASE}_clean_matches"
STAGE_PLAYERS = f"lakehouse.stage.{CASE}_clean_match_players"
STAGE_HEROES = f"lakehouse.stage.{CASE}_clean_heroes"

MART_MATCH = f"lakehouse.mart.{CASE}_match_draft_mart"
MART_PAIRS = f"lakehouse.mart.{CASE}_hero_matchups"

def table_exists(spark, name):
    try:
        return spark.catalog.tableExists(name)
    except Exception:
        return False

with SparkSessionContext(f"{CASE}-source-inspection") as spark:
    for name in [MATCHES_SOURCE, PLAYERS_SOURCE, HEROES_SOURCE]:
        if name and table_exists(spark, name):
            print(f"\n{name}")
            spark.table(name).printSchema()
            print("Строк:", spark.table(name).count())
        else:
            print(f"NOT FOUND: {name}")

## 3. RAW — загрузка исходных сущностей

Забираем только необходимые поля, но не применяем бизнес-фильтры. Список выбираемых колонок формируется динамически: это позволяет не падать, если необязательное поле (например, `cluster` или `rank`) отсутствует. Обязательные поля проверяются отдельно.

In [ ]:
def select_existing(df, mapping, required=()):
    """mapping: {alias: source_column}; выбирает только существующие поля."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Не найдены обязательные поля {missing}. Доступны: {df.columns}")
    exprs = [F.col(src).alias(alias) for alias, src in mapping.items() if src in df.columns]
    return df.select(*exprs)

with SparkSessionContext(f"{CASE}-raw") as ctx:
    spark = ctx

    matches_src = spark.table(MATCHES_SOURCE)
    match_mapping = {
        "match_id": "match_id",
        "start_time": "start_time",
        "duration": "duration",
        "radiant_win": "radiant_win",
        "human_players": "human_players",
        "lobby_type": "lobby_type",
        "game_mode": "game_mode",
        "cluster": "cluster",
        "loaded_at": "loaded_at",
        "sys_insert_stamp": "sys_insert_stamp"
    }
    raw_matches = select_existing(
        matches_src, match_mapping,
        required=("match_id", "radiant_win", "duration")
    )
    if "loaded_at" not in raw_matches.columns:
        raw_matches = raw_matches.withColumn("loaded_at", F.current_timestamp())
    if "sys_insert_stamp" not in raw_matches.columns:
        raw_matches = raw_matches.withColumn("sys_insert_stamp", F.current_timestamp())
    raw_matches.writeTo(RAW_MATCHES).using("iceberg").createOrReplace()
    print("RAW matches:", raw_matches.count())

    players_src = spark.table(PLAYERS_SOURCE)
    player_mapping = {
        "match_id": "match_id",
        "player_slot": "player_slot",
        "account_id": "account_id",
        "persona_name": "persona_name",
        "hero_id": "hero_id",
        "is_radiant": "is_radiant",
        "win": "win",
        "kills": "kills",
        "deaths": "deaths",
        "assists": "assists",
        "kda": "kda",
        "gold_per_min": "gold_per_min",
        "xp_per_min": "xp_per_min",
        "rank": "rank",
        "loaded_at": "loaded_at",
        "sys_insert_stamp": "sys_insert_stamp"
    }
    raw_players = select_existing(
        players_src, player_mapping,
        required=("match_id", "player_slot", "hero_id")
    )
    if "sys_insert_stamp" not in raw_players.columns:
        raw_players = raw_players.withColumn("sys_insert_stamp", F.current_timestamp())
    raw_players.writeTo(RAW_PLAYERS).using("iceberg").createOrReplace()
    print("RAW match_players:", raw_players.count())

    if HEROES_SOURCE and table_exists(spark, HEROES_SOURCE):
        heroes_src = spark.table(HEROES_SOURCE)
        hero_mapping = {
            "hero_id": "hero_id",
            "localized_name": "localized_name",
            "primary_attr": "primary_attr",
            "attack_type": "attack_type",
            "roles": "roles",
            "loaded_at": "loaded_at",
            "sys_insert_stamp": "sys_insert_stamp"
        }
        raw_heroes = select_existing(heroes_src, hero_mapping, required=("hero_id",))
        if "sys_insert_stamp" not in raw_heroes.columns:
            raw_heroes = raw_heroes.withColumn("sys_insert_stamp", F.current_timestamp())
        raw_heroes.writeTo(RAW_HEROES).using("iceberg").createOrReplace()
        print("RAW heroes:", raw_heroes.count())
    else:
        print("Справочник героев не загружен. MART будет построен без названий/атрибутов героев.")

## 4. STAGE — очистка и Data Funnel

Оставляем только технически корректные матчи и игроков. Для матча требуется ровно 10 уникальных слотов, 10 валидных героев и по 5 игроков на сторону. При конфликте стороны используем `player_slot < 128` как контрольный источник.

In [ ]:
funnel = []

def funnel_log(entity, step, reason, before, after):
    removed = before - after
    funnel.append({
        "Сущность": entity, "Этап": step, "Причина": reason,
        "Было": before, "Стало": after, "Удалено": removed,
        "Удалено (%)": round(removed / before * 100, 2) if before else 0.0
    })

with SparkSessionContext(f"{CASE}-stage") as ctx:
    spark = ctx

    # Матчи
    m = spark.table(RAW_MATCHES)
    n0 = m.count()
    m = m.withColumn("match_id", F.col("match_id").cast("long")) \
         .withColumn("duration", F.col("duration").cast("int")) \
         .withColumn("radiant_win", F.col("radiant_win").cast("boolean"))
    w = Window.partitionBy("match_id").orderBy(F.col("sys_insert_stamp").desc_nulls_last())
    m = m.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    n1 = m.count(); funnel_log("matches", "Дедупликация", "Повторные match_id", n0, n1)
    m = m.filter(
        F.col("match_id").isNotNull() & F.col("radiant_win").isNotNull() &
        F.col("duration").between(600, 7200)
    )
    n2 = m.count(); funnel_log("matches", "Валидация", "NULL ключ/цель или длительность вне 10–120 минут", n1, n2)
    m.writeTo(STAGE_MATCHES).using("iceberg").createOrReplace()

    # Игроки
    p = spark.table(RAW_PLAYERS)
    p0 = p.count()
    p = p.withColumn("match_id", F.col("match_id").cast("long")) \
         .withColumn("player_slot", F.col("player_slot").cast("int")) \
         .withColumn("hero_id", F.col("hero_id").cast("int"))
    wp = Window.partitionBy("match_id", "player_slot").orderBy(F.col("sys_insert_stamp").desc_nulls_last())
    p = p.withColumn("_rn", F.row_number().over(wp)).filter("_rn = 1").drop("_rn")
    p1 = p.count(); funnel_log("players", "Дедупликация", "Повторные записи (match_id, player_slot)", p0, p1)

    if "is_radiant" in p.columns:
        p = p.withColumn("is_radiant", F.coalesce(F.col("is_radiant").cast("boolean"), F.col("player_slot") < 128))
    else:
        p = p.withColumn("is_radiant", F.col("player_slot") < 128)

    p = p.filter(
        F.col("match_id").isNotNull() &
        F.col("player_slot").isin(0,1,2,3,4,128,129,130,131,132) &
        F.col("hero_id").isNotNull() & (F.col("hero_id") > 0)
    )
    p2 = p.count(); funnel_log("players", "Валидация слота и героя", "Некорректный player_slot или hero_id", p1, p2)

    valid_matches = spark.table(STAGE_MATCHES).select("match_id")
    p = p.join(valid_matches, "match_id", "inner")
    p3 = p.count(); funnel_log("players", "Ссылочная целостность", "Игроки без валидного матча", p2, p3)

    # Контроль состава: 10 уникальных слотов, 10 героев, 5 на каждой стороне
    match_counts = p.groupBy("match_id").agg(
        F.countDistinct("player_slot").alias("slots"),
        F.countDistinct("hero_id").alias("heroes"),
        F.sum(F.when(F.col("is_radiant"), 1).otherwise(0)).alias("radiant_n"),
        F.sum(F.when(~F.col("is_radiant"), 1).otherwise(0)).alias("dire_n")
    )
    good_ids = match_counts.filter(
        (F.col("slots") == 10) & (F.col("heroes") == 10) &
        (F.col("radiant_n") == 5) & (F.col("dire_n") == 5)
    ).select("match_id")
    p = p.join(good_ids, "match_id", "inner")
    p4 = p.count(); funnel_log("players", "Полный драфт 5×5", "Матчи не с 10 уникальными героями/слотами и составом 5×5", p3, p4)
    p.writeTo(STAGE_PLAYERS).using("iceberg").createOrReplace()

    # Удаляем из matches матчи, которые не прошли проверку состава
    m_clean = spark.table(STAGE_MATCHES).join(good_ids, "match_id", "inner")
    m_clean.writeTo(STAGE_MATCHES).using("iceberg").createOrReplace()

    # Справочник героев: дедупликация и проверка ключа
    if table_exists(spark, RAW_HEROES):
        h = spark.table(RAW_HEROES)
        h0 = h.count()
        wh = Window.partitionBy("hero_id").orderBy(F.col("sys_insert_stamp").desc_nulls_last())
        h = h.withColumn("_rn", F.row_number().over(wh)).filter("_rn = 1").drop("_rn")
        h1 = h.count(); funnel_log("heroes", "Дедупликация", "Повторные hero_id", h0, h1)
        h = h.filter(F.col("hero_id").isNotNull() & (F.col("hero_id") > 0))
        h2 = h.count(); funnel_log("heroes", "Валидация", "Пустой или некорректный hero_id", h1, h2)
        h.writeTo(STAGE_HEROES).using("iceberg").createOrReplace()

display(pd.DataFrame(funnel))

## 5. MART — признаки матча и таблица контр-пиков

`match_draft_mart` — одна строка на матч: 5 героев Radiant и 5 героев Dire в виде массивов, контекст и целевая переменная.  
`hero_matchups` — 25 строк на матч: каждая пара Radiant × Dire. Эта таблица подходит для расчёта статистики матчапов и анализа контр-пиков.

Не кодируем победителя в признаки состава: `radiant_win` хранится только как label.

In [ ]:
with SparkSessionContext(f"{CASE}-mart") as ctx:
    spark = ctx
    players = spark.table(STAGE_PLAYERS)
    matches = spark.table(STAGE_MATCHES)

    # Устойчивый порядок героев по player_slot; без pick order это лишь позиционный порядок,
    # а не хронология выбора.
    radiant = (players.filter("is_radiant = true")
        .groupBy("match_id")
        .agg(F.sort_array(F.collect_list(F.struct("player_slot", "hero_id"))).alias("r"))
        .select("match_id", F.expr("transform(r, x -> x.hero_id)").alias("radiant_heroes")))
    dire = (players.filter("is_radiant = false")
        .groupBy("match_id")
        .agg(F.sort_array(F.collect_list(F.struct("player_slot", "hero_id"))).alias("d"))
        .select("match_id", F.expr("transform(d, x -> x.hero_id)").alias("dire_heroes")))

    context_cols = [c for c in ["match_id","start_time","duration","radiant_win","lobby_type","game_mode","cluster","human_players"] if c in matches.columns]
    match_mart = (matches.select(*context_cols)
        .join(radiant, "match_id", "inner")
        .join(dire, "match_id", "inner")
        .withColumn("total_players", F.lit(10))
        .withColumn("target_radiant_win", F.col("radiant_win").cast("int"))
        .withColumn("duration_min", F.round(F.col("duration") / 60.0, 2))
        .withColumn("match_pace_bin",
            F.when(F.col("duration") < 1500, "fast")
             .when(F.col("duration") < 2400, "medium")
             .when(F.col("duration") < 3300, "long")
             .otherwise("very_long"))
    )

    # Метаданные героев — необязательное обогащение
    if table_exists(spark, STAGE_HEROES):
        heroes = spark.table(STAGE_HEROES)
        if "localized_name" in heroes.columns:
            hname = heroes.select("hero_id", F.col("localized_name").alias("hero_name"))
            match_mart = match_mart.withColumn("all_heroes", F.concat("radiant_heroes", "dire_heroes"))
            # Имена в отдельной нормализованной таблице ниже, чтобы не усложнять массивы.
    match_mart.writeTo(MART_MATCH).using("iceberg").createOrReplace()

    # 5×5 cross join пар героев в пределах одного матча
    r_long = players.filter("is_radiant = true").select("match_id", F.col("hero_id").alias("radiant_hero_id"))
    d_long = players.filter("is_radiant = false").select("match_id", F.col("hero_id").alias("dire_hero_id"))
    pairs = (r_long.join(d_long, "match_id", "inner")
        .join(matches.select("match_id", "radiant_win"), "match_id", "inner")
        .withColumn("target_radiant_win", F.col("radiant_win").cast("int"))
        .select("match_id", "radiant_hero_id", "dire_hero_id", "target_radiant_win"))
    pairs.writeTo(MART_PAIRS).using("iceberg").createOrReplace()

    print("Матчей в MART:", spark.table(MART_MATCH).count())
    print("Пар героев в MART:", spark.table(MART_PAIRS).count())
    spark.table(MART_MATCH).show(5, truncate=False)
    spark.table(MART_PAIRS).show(10, truncate=False)

## 6. Data Quality — контроль целостности витрин

Проверяем уникальность матчей, размер составов, уникальность героев внутри матча, допустимость target и ожидаемое число пар (25 на матч). Нулевое число нарушений — обязательное условие перед ML.

In [ ]:
with SparkSessionContext(f"{CASE}-quality") as spark:
    checks = {
        "Уникальность match_id в STAGE matches":
            f"select count(*) - count(distinct match_id) as n from {STAGE_MATCHES}",
        "Уникальность match_id в MART":
            f"select count(*) - count(distinct match_id) as n from {MART_MATCH}",
        "Целевая переменная только 0/1":
            f"select count_if(target_radiant_win not in (0,1) or target_radiant_win is null) as n from {MART_MATCH}",
        "Ровно 5 героев у каждой стороны":
            f"select count_if(size(radiant_heroes) <> 5 or size(dire_heroes) <> 5) as n from {MART_MATCH}",
        "25 пар героев на матч":
            f"select count_if(pair_cnt <> 25) as n from (select match_id, count(*) pair_cnt from {MART_PAIRS} group by match_id)",
        "Нет зеркальных/повторных героев в драфте":
            f"select count_if(size(array_distinct(concat(radiant_heroes,dire_heroes))) <> 10) as n from {MART_MATCH}"
    }
    for label, query in checks.items():
        value = spark.sql(query).first()["n"]
        print(f"{label}: {'PASS' if value == 0 else 'FAIL'} ({value})")

    print("Сводка:")
    spark.table(MART_MATCH).select(
        F.count("*").alias("matches"),
        F.avg("target_radiant_win").alias("radiant_win_rate"),
        F.avg("duration_min").alias("avg_duration_min")
    ).show(truncate=False)

## 7. Подготовка ML-датасета и защита от leakage

Для первой baseline-модели используем multi-hot представление 10 выбранных героев: по каждому `hero_id` создаётся бинарный признак присутствия в составе Radiant и Dire. Для симметричной модели можно дополнительно перейти к признакам `hero_present` без привязки к стороне, но текущая постановка предсказывает именно победу Radiant.

Разбиение train/test должно быть временным по `start_time`. Любые target encoding, win rate героя и pair win rate рассчитываются только внутри train (лучше out-of-fold), затем применяются к validation/test.

In [ ]:
# Spark -> Pandas для baseline. При большом объёме данных используйте Spark ML,
# а не toPandas() на всей таблице.
with SparkSessionContext(f"{CASE}-ml-export") as spark:
    ml_pd = spark.table(MART_MATCH).select(
        "match_id", "start_time", "radiant_heroes", "dire_heroes",
        "target_radiant_win", "duration_min", "game_mode", "lobby_type"
    ).toPandas()

print("Размер:", ml_pd.shape)
display(ml_pd.head())

# Контроль: не используем post-match признаки игроков (KDA, GPM, XPM, итоговый rank),
# если задача — предсказать победителя до/во время драфта.

## 8. Итог и следующие шаги

После выполнения ноутбука должны быть доступны:
- `lakehouse.raw.practice_case_22_matches`
- `lakehouse.raw.practice_case_22_match_players`
- `lakehouse.raw.practice_case_22_heroes` (если найден справочник)
- `lakehouse.stage.practice_case_22_clean_matches`
- `lakehouse.stage.practice_case_22_clean_match_players`
- `lakehouse.stage.practice_case_22_clean_heroes` (если найден справочник)
- `lakehouse.mart.practice_case_22_match_draft_mart`
- `lakehouse.mart.practice_case_22_hero_matchups`

**Перед запуском на полном объёме:** сначала проверь фактические имена и схемы источников в разделе 2. В частности, `HEROES_SOURCE` и наличие полей `is_radiant`, `player_slot`, `radiant_win`, `loaded_at` могут отличаться от ожидаемых. Код намеренно останавливается на отсутствующих обязательных полях, а не подменяет их догадками.

Дальнейший этап отдельным ноутбуком/разделом: временное train/test-разбиение, baseline (Logistic Regression / CatBoost), метрики ROC-AUC, PR-AUC, accuracy, calibration, permutation importance и анализ парных контр-пиков с доверительными интервалами.